In [25]:
import pandas as pd

file_paths = [
    r"C:\Users\lulay\Desktop\icoict26-challenge\literature_review\database-scopus\export_f4aef050-ef1f-4de4-aff5-e96625ef11a7_2026-04-20T120609.979573742.csv",
    r"C:\Users\lulay\Desktop\icoict26-challenge\literature_review\database-scopus\export_b8becafd-a5da-4c0a-8588-9a44d11ed116_2026-04-20T120615.190421722.csv",
    r"C:\Users\lulay\Desktop\icoict26-challenge\literature_review\database-scopus\export_f135aed2-d01c-4b40-9ea0-1ad7c9b2299d_2026-04-20T120555.945116225.csv"
]

df_raw = pd.concat([pd.read_csv(f) for f in file_paths], ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=['DOI', 'Title']).reset_index(drop=True)

print(f"Total Unique Records: {len(df_raw)}")
df_raw[['Authors', 'Title', 'Year']].head()

Total Unique Records: 585


,Authors,Title,Year
0,Thanh T.T.M.; Ly H.-B.; Pham B.T.,A possibility of AI application on mode-choice...,2020
1,NaN,22nd International Conference on Intelligent S...,2023
2,Zhou T.; Li Q.; Wang R.; Sun Y.; Zhang S.,Exploring effects of expressway accident on dr...,2025
3,Guo Q.; Mu L.; Lou S.,Revolutionizing travel experiences: An in-dept...,2024
4,Yöndem M.T.; Özçelik Ş.T.; Caetano I.; Figueir...,Transforming Tourism Experience: AI-Based Smar...,2023


In [26]:
# Constraints: 2021-2027, English, Peer-reviewed (Article/Conference)
mask = (
    (df_raw['Year'] >= 2021) & (df_raw['Year'] <= 2027) &
    (df_raw['Language of Original Document'].str.contains('English', na=False, case=False)) &
    (df_raw['Document Type'].isin(['Article', 'Conference Paper', 'Review']))
)

df_meta = df_raw[mask].copy()

print("Metadata Filter Results:")
display(df_meta['Document Type'].value_counts().to_frame(name='Count'))
print(f"Remaining Papers: {len(df_meta)}")

Metadata Filter Results:


,Count
Document Type,
Article,228
Review,9


Remaining Papers: 237


In [27]:
dl_keywords = ['deep learning', 'neural network', 'cnn', 'rnn', 'transformer', 'lstm', 'gan', 'reinforcement learning']
tourism_keywords = ['tourism', 'tourist', 'travel', 'hospitality', 'destination', 'hotel']

def is_relevant(row):
    text = f"{row['Title']} {row['Abstract']} {row['Author Keywords']}".lower()
    return any(k in text for k in dl_keywords) and any(k in text for k in tourism_keywords)

df_filtered = df_meta[df_meta.apply(is_relevant, axis=1)].copy()

print(f"Relevant DL & Tourism Papers: {len(df_filtered)}")

Relevant DL & Tourism Papers: 139


In [28]:
# Identify existing Literature Reviews vs Primary Research
df_filtered['Category'] = df_filtered['Document Type'].apply(
    lambda x: 'Literature Review' if x == 'Review' else 'Primary Research'
)

# Optional: Further check titles for "Review" keywords
review_keywords = ['review', 'survey', 'systematic', 'meta-analysis', 'state-of-the-art']
df_filtered.loc[
    (df_filtered['Category'] == 'Primary Research') & 
    (df_filtered['Title'].str.contains('|'.join(review_keywords), case=False, na=False)), 
    'Category'
] = 'Literature Review'

summary_table = df_filtered.groupby('Category').size().to_frame(name='Count')
display(summary_table)

,Count
Category,
Literature Review,11
Primary Research,128


In [29]:
# Sorting by impact
df_final = df_filtered.sort_values(by=['Category', 'Cited by'], ascending=[True, False])

# Displaying Top Literature Reviews first
print("Top Identified Literature Reviews:")
display(df_final[df_final['Category'] == 'Literature Review'][['Authors', 'Title', 'Year', 'Cited by']].head(10))

Top Identified Literature Reviews:


,Authors,Title,Year,Cited by
95,Sadeghian P.; Håkansson J.; Zhao X.,Review and evaluation of methods in transport ...,2021,56
430,Wu D.C.; Zhong S.; Qiu R.T.R.; Wu J.,Are customer reviews just reviews? Hotel forec...,2022,47
427,Zhang C.; Tian Y.-X.; Hu A.-Y.,Utilizing textual data from online reviews for...,2025,14
473,Dowlut N.; Gobin-Rahimbux B.,Forecasting resort hotel tourism demand using ...,2023,14
136,Ophir Y.; Tikochinski R.; Brunstein Klomek A.;...,The Hitchhiker’s Guide to Computational Lingui...,2022,13
418,Villar J.R.N.; Lengua M.A.C.,A Systematic Review of the Literature on the U...,2024,5
284,Budhwani A.; Lin T.; Feng D.; Bachmann C.,Assessing and Comparing Data Imputation Techni...,2023,3
181,Pathak R.C.; Agarwal P.; Gehlot A.; Singh R.; ...,Advanced Digital Technologies for Promoting In...,2025,2
549,Prasad A.V.; Vedavathi K.,FoodABSANet: Developing an adaptive graph conv...,2026,1
434,Seyam A.; Mathew S.S.; Barachi M.E.; Zhang C.;...,The application of machine learning and deep l...,2026,0


In [30]:
df_reviews_only = df_final[df_final['Category'] == 'Literature Review']
df_papers_only = df_final[df_final['Category'] == 'Primary Research']

# Export 1: Literature Reviews
df_reviews_only.to_csv(r"C:\Users\lulay\Desktop\icoict26-challenge\literature_review\filtered_literature_review_results.csv", index=False)

# Export 2: Primary Research Papers
df_papers_only.to_csv(r"C:\Users\lulay\Desktop\icoict26-challenge\literature_review\filtered_paper_results.csv", index=False)

In [31]:
print("Export Complete:")
print(f"- Saved {len(df_reviews_only)} reviews to filtered_literature_review_results.csv")
print(f"- Saved {len(df_papers_only)} papers to filtered_paper_results.csv")

Export Complete:
- Saved 11 reviews to filtered_literature_review_results.csv
- Saved 128 papers to filtered_paper_results.csv
